In [34]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import os
import random
# import tensorflow_model_optimization as tfmot
from metrics_tracking import F1Score, plot_metrics
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import random
import os
import struct

# Metrics Before Compression:

Test AUC: 0.8723267912864685 (most important)

Test Precision: 1.0

Test Recall: 0.5199999809265137

Test F1: 0.684210479259491

Test loss: 0.08323828130960464

Test accuracy: 0.9896759390830994

In [ ]:
#load dataset for model evaluation
#import the datasets to test



def load_data():
    data = np.load("Preprocessed_Data/roads_canids_windows_200hz_3s.npz")
    # Access arrays by their keys
    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test  = data["X_test"]
    y_test  = data["y_test"]
    X_train = X_train[..., :-1] #TEMP FIX REMOVE LATER - FIX PREPROCESSING TO GET RID OF STRING COLUMN
    X_test  = X_test[..., :-1] #TEMP FIX REMOVE LATER
    print(X_train.shape)
    print(y_train.shape)
    print(X_test.shape)
    print(y_test.shape)
    data.close()
    return X_train, y_train, X_test, y_test

def standardize(X_train, y_train, X_test, y_test):
    # Clip outliers 
    X_train = np.clip(X_train, -1e6, 1e6)
    X_test  = np.clip(X_test,  -1e6, 1e6)
    # Standardize features
    scaler = StandardScaler()
    X_train_flat = X_train.reshape(-1, X_train.shape[-1])
    X_test_flat  = X_test.reshape(-1,  X_test.shape[-1])
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_test_scaled  = scaler.transform(X_test_flat)
    X_train = X_train_scaled.reshape(X_train.shape)
    X_test  = X_test_scaled.reshape(X_test.shape)
    print("Scaler mean: ", scaler.mean_, "Scaler scale: ", scaler.scale_) 
    return X_train, y_train, X_test, y_test, scaler.mean_, scaler.scale_

X_train, y_train, X_test, y_test = load_data()
X_train, y_train, X_test, y_test, scaler_mean, scaler_std = standardize(X_train, y_train, X_test, y_test)

(11280, 600, 23)
(11280,)
(13948, 600, 23)
(13948,)


In [51]:
# SCALE = 1000.0  # float → int16 multiplier
def create_balanced_samples(X_test, y_test, per_class=1):
    attack_idx = np.where(y_test == 1)[0]
    normal_idx = np.where(y_test == 0)[0]
    chosen = random.sample(list(attack_idx), per_class) + \
             random.sample(list(normal_idx), per_class)
    random.shuffle(chosen)
    X_sel = X_test[chosen]
    y_sel = y_test[chosen]
    return X_sel, y_sel

In [49]:
OUTPUT_DIR = "Preprocessed_Data/STM_SAMPLES"
PREFIX = "samples_fp32"
SAMPLE_COUNT = 2  # number of samples to export

def write_samples_files_fp32(X_sel, y_sel, scaler_mean, scaler_std, prefix="samples_fp32"):
    os.makedirs("Preprocessed_Data/STM_SAMPLES", exist_ok=True)
    h_file = f"Preprocessed_Data/STM_SAMPLES/{prefix}.h"
    c_file = f"Preprocessed_Data/STM_SAMPLES/{prefix}.c"

    S, T, F = X_sel.shape

    # --------- HEADER FILE ---------
    with open(h_file, "w") as h:
        h.write("#pragma once\n")
        h.write("#include <stdint.h>\n\n")
        h.write(f"#define SAMPLE_COUNT {S}\n")
        h.write(f"#define SAMPLE_TIME   {T}\n")
        h.write(f"#define SAMPLE_FEATS  {F}\n\n")
        h.write(f"extern const float samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS];\n")
        h.write(f"extern const int   samples_y[SAMPLE_COUNT];\n")
        h.write(f"extern const float scaler_mean[SAMPLE_FEATS];\n")
        h.write(f"extern const float scaler_std[SAMPLE_FEATS];\n")

    # --------- C FILE ---------
    with open(c_file, "w") as c:
        c.write(f'#include "{prefix}.h"\n\n')

        # Samples
        c.write("const float samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS] = {\n")
        for s in range(S):
            c.write("  {\n")
            for t in range(T):
                row = ", ".join(f"{v:.6f}f" for v in X_sel[s, t])
                c.write(f"    {{ {row} }},\n")
            c.write("  },\n")
        c.write("};\n\n")

        # Labels
        c.write("const int samples_y[SAMPLE_COUNT] = { ")
        c.write(", ".join(str(int(v)) for v in y_sel))
        c.write(" };\n\n")

        # Scaler mean
        c.write("const float scaler_mean[SAMPLE_FEATS] = { ")
        c.write(", ".join(f"{v:.6f}f" for v in scaler_mean))
        c.write(" };\n\n")

        # Scaler std
        c.write("const float scaler_std[SAMPLE_FEATS] = { ")
        c.write(", ".join(f"{v:.6f}f" for v in scaler_std))
        c.write(" };\n")

    print(f"Generated {h_file} and {c_file} successfully.")


In [52]:
# X_sel, y_sel = create_balanced_csv_samples(X_test, y_test, per_class=1)
# write_samples_files_fp16_safe(X_sel, y_sel, prefix="samples_fp16")
 # Select balanced subset
X_sel, y_sel = create_balanced_samples(X_test, y_test, per_class=1)
# Generate FP32 .h/.c
write_samples_files_fp32(X_sel, y_sel, scaler_mean, scaler_std, prefix="samples_fp32")

Generated Preprocessed_Data/STM_SAMPLES/samples_fp32.h and Preprocessed_Data/STM_SAMPLES/samples_fp32.c successfully.


In [ ]:
def write_samples_files_fp32_raw(X_sel, y_sel, prefix="samples_fp32"):
    S, T, F = X_sel.shape

    folder = "Preprocessed_Data/STM_SAMPLES"
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, f"{prefix}.h"), "w") as h:
        h.write("#pragma once\n")
        h.write("#include <stdint.h>\n\n")
        h.write(f"#define SAMPLE_COUNT {S}\n")
        h.write(f"#define SAMPLE_TIME   {T}\n")
        h.write(f"#define SAMPLE_FEATS  {F}\n\n")
        h.write("extern const float samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS];\n")
        h.write("extern const int samples_y[SAMPLE_COUNT];\n")
    with open(os.path.join(folder, f"{prefix}.c"), "w") as c:
        c.write(f'#include "{prefix}.h"\n')
        c.write("#include <stdint.h>\n\n")
        c.write("const float samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS] = {\n")
        for s in range(S):
            c.write("  {\n")
            for t in range(T):
                row = ", ".join(f"{v:.8f}f" for v in X_sel[s, t])
                c.write(f"    {{ {row} }},\n")
            c.write("  },\n")
        c.write("};\n\n")
        c.write("const int samples_y[SAMPLE_COUNT] = { ")
        c.write(", ".join(str(int(v)) for v in y_sel))
        c.write(" };\n")

In [32]:
X_sel, y_sel = create_balanced_csv_samples(X_test, y_test, per_class=1)
write_samples_files_fp32_raw(X_sel, y_sel, prefix="samples_fp32_single")

In [5]:

SCALE = 1.0  # FP16 stores raw float values

def float_to_fp16(val):
    """Convert float32 to IEEE 754 half-precision (uint16)."""
    s = struct.pack('e', val)  # half-precision binary
    return struct.unpack('H', s)[0]

def write_samples_files_fp16_from_csv(csv_files, y_csv, prefix="samples"):
    """
    csv_files: list of CSV paths, each containing one sample of shape [T, F]
    y_csv: CSV containing labels (1D array)
    """
    X_list = []
    for f in csv_files:
        X = np.loadtxt(f, delimiter=',', dtype=np.float32)
        X_list.append(X)
    X_sel = np.stack(X_list, axis=0)  # shape: [S, T, F]
    y_sel = np.loadtxt(y_csv, dtype=int)  # shape: [S]

    S, T, F = X_sel.shape
    os.makedirs("Preprocessed_Data", exist_ok=True)

    # Write header
    with open(f"Preprocessed_Data/{prefix}.h", "w") as h:
        h.write("#pragma once\n\n")
        h.write(f"#define SAMPLE_COUNT {S}\n#define SAMPLE_TIME {T}\n#define SAMPLE_FEATS {F}\n#define SAMPLE_SCALE {SCALE}f\n\n")
        h.write("extern const uint16_t samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS];\n")
        h.write("extern const int samples_y[SAMPLE_COUNT];\n")

    # Write C file
    with open(f"Preprocessed_Data/{prefix}.c", "w") as c:
        c.write(f'#include "{prefix}.h"\n\n')
        c.write("const uint16_t samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS] = {\n")
        for s in range(S):
            c.write("  {\n")
            for t in range(T):
                row = ", ".join(str(float_to_fp16(float(v))) for v in X_sel[s, t])
                c.write(f"    {{ {row} }},\n")
            c.write("  },\n")
        c.write("};\n\n")
        c.write("const int samples_y[SAMPLE_COUNT] = { " + ", ".join(str(int(v)) for v in y_sel) + " };\n")

In [6]:
X_sel, y_sel = create_balanced_csv_samples(X_test, y_test, per_class=2)
write_samples_files_fp16_from_csv(X_sel, y_sel, prefix="samples_int16")

TypeError: non-string returned while reading data

In [ ]:
# keras_model = keras.models.load_model( #import model for quantization
#     "best_ROAD_model128_F1Fixed.keras",
#     compile=True,
#     custom_objects={"F1Score": F1Score},
#     safe_mode=False
)
keras_model.summary()

ValueError: File not found: filepath=best_ROAD_model128_F1Fixed.keras. Please ensure the file is an accessible `.keras` zip file.

In [72]:
#code here on out borrowed from ECE528 Assignment3 (Question 2/3)
def get_gzipped_model_size(keras_model, fname):
  # It returns the size of the gzipped model in bytes.
  import os
  import zipfile
  import tempfile
  keras_model.save(fname)
  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(fname, arcname=os.path.basename(fname))
  return os.path.getsize(zipped_file)


gzipped_size = get_gzipped_model_size(keras_model, "best_ROAD_model128_F1Fixed.keras")
print("Gzipped uncompressed model size (bytes):", gzipped_size)

Gzipped uncompressed model size (bytes): 273358


In [73]:
TFLITE_FP32 = "road_128_cnn.tflite"
TFLITE_QUANT_DYNAMIC = "road_128_cnn_quant_dyn.tflite"

X_test = X_test.astype(np.float32)
print("x_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

x_test shape: (13948, 600, 23)
y_test shape: (13948,)


In [74]:
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
tflite_model = converter.convert()
with open(TFLITE_FP32, "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpipumh7ia'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 600, 23), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136932460168208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460168976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460172240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171856: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [75]:
#cconvert to tflite with dynamic conversion
dynamic_converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
dynamic_converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant = converter.convert()
with open(TFLITE_QUANT_DYNAMIC, "wb") as f:
    f.write(tflite_quant)

Saved artifact at '/tmp/tmpo696s6j1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 600, 23), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136932460168208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460168976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460172240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171856: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
def representative_dataset():
    indices = np.arange(len(X_train))
    np.random.shuffle(indices)
    for i in indices[:500]:  # 500 random windows
        x = X_train[i:i+1].astype(np.float32)
        yield [x]

TFLITE_INT8 = "road_128_cnn_INT8.tflite"
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8] ## converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_int8 = converter.convert()
with open(TFLITE_INT8, "wb") as f:
    f.write(tflite_int8)

Saved artifact at '/tmp/tmpek2msxc1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 600, 23), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136932460168208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460168976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460172240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171856: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [85]:

def evaluate_tflite_model_roc(model_path, x_data, y_labels, clip_range=(-1.0, 1.0)):
    """
    Evaluate a TFLite model (FP32 or INT8) and return ROC-AUC.
    Automatically handles input/output quantization and clipping for INT8.
    """
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    n = x_data.shape[0]
    y_probs = np.zeros(n, dtype=np.float32)
    input_dtype  = input_details[0]['dtype']
    output_dtype = output_details[0]['dtype']
    in_scale, in_zero = input_details[0]['quantization']
    out_scale, out_zero = output_details[0]['quantization']
    for i in range(n):
        sample = x_data[i:i+1].astype(np.float32)
        # Clip inputs to safe range
        sample = np.clip(sample, clip_range[0], clip_range[1])
        # Quantize if INT8 input
        if input_dtype == np.int8:
            sample = (sample / in_scale + in_zero).astype(np.int8)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        # Dequantize if INT8 output
        if output_dtype == np.int8:
            output = (output.astype(np.float32) - out_zero) * out_scale
        # Binary classifier: probability of "Attack"
        y_probs[i] = output[0][0]
    auc = roc_auc_score(y_labels, y_probs)
    return auc

In [87]:
auc_int8 = evaluate_tflite_model_roc("road_int8.tflite", X_test, y_test)
print(f"INT8 TFLite ROC-AUC: {auc_int8:.4f}")

INT8 TFLite ROC-AUC: 0.5000


In Summary: INT8 Quantization fails bc completely removes effectiveness of model.
